# OSeMOSYS for Sierra Leone - Long Term Planning with Seasonality

## Setup and Pre-Processing

In [10]:
import os
from pathlib import Path

# === USER INPUT ===
model_name = "Scenario 2 - Optimal Capacity Expansion"  
# ===================

# === Set Dynamic File Paths ===
base_dir = Path("models") / model_name
data_file = base_dir / "data.txt"
model_file = base_dir / "model.txt"
solution_file = base_dir / "solution.txt"

# Output folders
res_dir = Path("res/csv")

# Ensure output directories exist
solution_file.parent.mkdir(parents=True, exist_ok=True)
res_dir.mkdir(parents=True, exist_ok=True)

## Running the Model

In [11]:
import subprocess

# Command to run GLPK
cmd = ["glpsol", "-m", model_file, "-d", data_file, "-o", solution_file]

# Run and capture output
try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print("Solver ran successfully!")
    print(result.stdout)  # See solver output
except subprocess.CalledProcessError as e:
    print("Solver failed!")
    print(e.stdout)       # Show solver standard output
    print(e.stderr)       # Show solver error log


Solver ran successfully!
GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 -m models\Scenario 2 - Optimal Capacity Expansion\model.txt -d models\Scenario 2 - Optimal Capacity Expansion\data.txt
 -o models\Scenario 2 - Optimal Capacity Expansion\solution.txt
Reading model section from models\Scenario 2 - Optimal Capacity Expansion\model.txt...
185 lines were read
Reading data section from models\Scenario 2 - Optimal Capacity Expansion\data.txt...
695 lines were read
Generating OFL_Cost...
Generating Acc1_FuelProductionByTechnology...
Generating Acc2_FuelUseByTechnology...
Generating Acc3_AverageAnnualRateOfActivity...
Generating CAa1_TotalNewCapacity...
Generating CAa2_TotalAnnualCapacity...
Generating CAa5_TotalNewCapacity...
Generating CC1_UndiscountedCapitalInvestment...
Generating E2_AnnualEmissionProduction...
Generating EBa10_EnergyBalanceEachTS4...
Generating EBa1_RateOfFuelProduction1...
Generating EBa2_RateOfFuelProduction2...
Generating EBa4_RateOf

## Post Processing

In [12]:
# === Helper Function ===
def run_command(cmd, success_msg, fail_msg, exit_on_fail=False, ignore_errors=False):
    print(f"\n🔄 Running: {' '.join(map(str, cmd))}")
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(f"✅ {success_msg}")
        return result.stdout
    except subprocess.CalledProcessError as e:
        if ignore_errors:
            print(f"⚠️ Partial success: {fail_msg} — continuing anyway.")
            print("STDOUT:\n", e.stdout)
            return e.stdout
        else:
            print(f"❌ {fail_msg}")
            print("STDOUT:\n", e.stdout)
            print("STDERR:\n", e.stderr)
            if exit_on_fail:
                exit(1)
            return None

# === Step 1: Setup config.yaml ===
if not os.path.exists("config.yaml"):
    run_command(
        ["otoole", "setup", "config", "config.yaml"],
        "✅ config.yaml created successfully!",
        "❌ Failed to generate config.yaml",
        exit_on_fail=True
    )
else:
    run_command(
        ["otoole", "setup", "config", "config.yaml", "--overwrite"],
        "✅ config.yaml overwritten successfully!",
        "❌ Failed to overwrite config.yaml"
    )

# === Step 2: Validate model ===
run_command(
    ["otoole", "validate", "datafile", str(data_file), "config.yaml"],
    "✅ Model data validated.",
    "⚠️ Validation warnings or errors occurred."
)

# === Step 3A: Convert data.txt (inputs) to CSV ===
run_command(
    [
        "otoole", "convert", "datafile", "csv",
        str(data_file), str(res_dir), "config.yaml"
    ],
    f"✅ Inputs extracted to: {res_dir}",
    "❌ Failed to convert inputs"
)

# === Step 3B: Extract solution.txt (outputs) to CSV ===
run_command(
    [
        "otoole", "results", "glpk", "csv",
        str(solution_file),
        str(res_dir),
        "datafile",
        str(data_file),
        "config.yaml",
        "--glpk_model", str(model_file)
    ],
    f"✅ Results extracted to: {res_dir}",
    "⚠️ Some results could not be extracted",
    ignore_errors=True  # Let it continue even if some result vars fail
)

print(f"\n🎉 Full pipeline completed for model: {model_name}")


🔄 Running: otoole setup config config.yaml --overwrite
✅ ✅ config.yaml overwritten successfully!

🔄 Running: otoole validate datafile models\Scenario 2 - Optimal Capacity Expansion\data.txt config.yaml
✅ ✅ Model data validated.

🔄 Running: otoole convert datafile csv models\Scenario 2 - Optimal Capacity Expansion\data.txt res\csv config.yaml
✅ ✅ Inputs extracted to: res\csv

🔄 Running: otoole results glpk csv models\Scenario 2 - Optimal Capacity Expansion\solution.txt res\csv datafile models\Scenario 2 - Optimal Capacity Expansion\data.txt config.yaml --glpk_model models\Scenario 2 - Optimal Capacity Expansion\model.txt
⚠️ Partial success: ⚠️ Some results could not be extracted — continuing anyway.
STDOUT:
 ParserError: Error tokenizing data. C error: Expected 5 fields in line 18, saw 10



🎉 Full pipeline completed for model: Scenario 2 - Optimal Capacity Expansion


## Results Visualization

### Colors

In [13]:
import pandas as pd
import json

# === Step 1: Load technologies from CSV ===
tech_file = res_dir / "TECHNOLOGY.csv"  
df_tech = pd.read_csv(tech_file)
technologies = sorted(df_tech["VALUE"].dropna().unique())  

# === Step 2: Assign colors using standard energy palette ===
tech_colors = {}

for tech in technologies:
    tech_lower = tech.lower()

    # Explicit colors for named hydro technologies
    if tech == "Hydro_BumbunaI" or tech == "Hydro_Medium":
        tech_colors[tech] = "#1f77b4"  # Dark blue
    elif tech == "Hydro_BumbunaII" or tech == "Hydro_Large":
        tech_colors[tech] = "#0055A4"  # Deeper blue
    elif tech == "Hydro_Bekongor":
        tech_colors[tech] = "#4da6ff"  # Medium blue
    elif tech == "Hydro_Betmai":
        tech_colors[tech] = "#87CEFA"  # Light blue
    elif tech == "Hydro_Singimi" or tech == "Hydro_Small":
        tech_colors[tech] = "#ADD8E6"  # Very light blue

    elif "solar" in tech_lower:
        tech_colors[tech] = "#FFD700"  # Yellow
    elif "karpowership" in tech_lower or "hfo" in tech_lower:
        tech_colors[tech] = "#000000"  # Black
    elif "biomass" in tech_lower:
        tech_colors[tech] = "#228B22"  # Green
    elif "lfo" in tech_lower:
        tech_colors[tech] = "#8B4513"  # Brown
    elif "_prod" in tech_lower:
        tech_colors[tech] = "#A9A9A9"  # Gray (fuel producers)
    else:
        tech_colors[tech] = "#D3D3D3"  # Light gray fallback

# === Step 3: Print & export ===
print("\n🎨 Assigned technology colors:")
for tech, color in tech_colors.items():
    print(f"{tech}: {color}")

# Save to JSON
with open("tech_colors.json", "w") as f:
    json.dump(tech_colors, f, indent=2)


🎨 Assigned technology colors:
Biomass_Plant: #228B22
Biomass_Prod: #228B22
HFO_Plant: #000000
HFO_Prod: #000000
Hydro_Bekongor: #4da6ff
Hydro_Betmai: #87CEFA
Hydro_BumbunaI: #1f77b4
Hydro_BumbunaII: #0055A4
Hydro_Large: #0055A4
Hydro_Medium: #1f77b4
Hydro_Singimi: #ADD8E6
Hydro_Small: #ADD8E6
Karpowership: #000000
LFO_Plant: #8B4513
LFO_Prod: #8B4513
Solar_PV: #FFD700


### Total Annual Installed Capacity

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# === Load total and residual capacity data ===
df_total = pd.read_csv(res_dir / "TotalCapacityAnnual.csv")
df_resid = pd.read_csv(res_dir / "ResidualCapacity.csv")

# Rename columns
df_total.rename(columns={"t": "TECHNOLOGY", "y": "YEAR", "TotalCapacityAnnual": "VALUE"}, inplace=True)
df_resid.rename(columns={"t": "TECHNOLOGY", "y": "YEAR", "ResidualCapacity": "VALUE"}, inplace=True)

# Pivot for plotting
df_total = df_total.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
df_resid = df_resid.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
df_new = df_total.sub(df_resid, fill_value=0).clip(lower=0)
years = df_total.index.tolist()

# === Load tech colors ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Create plot ===
fig = go.Figure()
trace_names = []

# === Define technologies to exclude from legend ===
exclude_from_legend = ["Hydro_Bekongor", "Hydro_Betmai", "Hydro_BumbunaI", "Hydro_BumbunaII", "Hydro_Singimi","Karpowership"]

# Add residual capacity bars (patterned)
for tech in df_resid.columns:
    trace_name = f"{tech} (residual)"
    trace_names.append(trace_name)
    fig.add_trace(go.Bar(
        x=years,
        y=df_resid[tech].fillna(0),
        name=trace_name,
        marker=dict(
            color=tech_colors.get(tech, "#D3D3D3"),
            pattern_shape="/"
        ),
        hovertemplate=f"{tech} (residual)<br>Year: %{{x}}<br>Capacity: %{{y}} GW"
    ))

# Add new capacity bars (solid)
for tech in df_new.columns:
    trace_name = tech
    trace_names.append(trace_name)
    fig.add_trace(go.Bar(
        x=years,
        y=df_new[tech].fillna(0),
        name=trace_name,
        showlegend=tech not in exclude_from_legend,
        marker=dict(color=tech_colors.get(tech, "#D3D3D3")),
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>New Capacity: %{{y}} GW"
    ))

# === Dropdown visibility filters ===
visible_all = [True] * len(trace_names)
visible_residual_only = [name.endswith("(residual)") for name in trace_names]
visible_new_only = [not name.endswith("(residual)") for name in trace_names]
visible_no_prod = [
    not ("_Prod" in name.replace(" (residual)", "")) for name in trace_names
]

# === Final layout ===
fig.update_layout(
    title="Installed Capacity by Technology",
    xaxis_title="Year",
    yaxis_title="Capacity (GW)",
    barmode='stack',
    template="plotly_white",
    height=600,
    legend_title="Technology",
    hovermode="x unified",
    updatemenus=[{
        "buttons": [
            {"label": "All", "method": "update", "args": [{"visible": visible_all}]},
            {"label": "Residual Only", "method": "update", "args": [{"visible": visible_residual_only}]},
            {"label": "New Capacity Only", "method": "update", "args": [{"visible": visible_new_only}]},
            {"label": "Exclude _Prod", "method": "update", "args": [{"visible": visible_no_prod}]}
        ],
        "direction": "down",
        "showactive": True,
        "x": 1.05,
        "xanchor": "left",
        "y": 1.15,
        "yanchor": "top"
    }]
)

fig.show()


In [28]:
# Create a sizing table: rows = technologies, columns = years, values in MW
df_sizing_mw = (df_total.T * 1000).round(1)  # Convert GW to MW and round

# Set proper row and column labels
df_sizing_mw.index.name = "Technology (MW)"
df_sizing_mw.columns.name = "Year"

# Display the table
display(df_sizing_mw)


Year,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
Technology (MW),,,,,,,,,,,,,,,,,,,,,
Biomass_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Biomass_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HFO_Plant,94.4,94.4,94.4,94.4,94.4,119.4,119.4,119.4,119.4,119.4,...,111.2,111.2,111.2,111.2,111.2,111.2,111.2,111.2,111.2,50.0
HFO_Prod,3326.3,3326.3,3326.3,3326.3,3326.3,3326.3,3326.3,3326.3,3326.3,3326.3,...,5709.0,5709.0,5709.0,5709.0,5709.0,5709.0,5709.0,5709.0,5709.0,5709.0
Hydro_Bekongor,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,160.0,160.0,160.0,160.0,160.0,160.0,160.0,160.0,160.0,160.0
Hydro_Betmai,0.0,0.0,0.0,0.0,0.0,27.0,27.0,27.0,27.0,27.0,...,27.0,27.0,27.0,27.0,27.0,27.0,27.0,27.0,27.0,27.0
Hydro_BumbunaI,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,...,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
Hydro_BumbunaII,0.0,0.0,0.0,0.0,0.0,143.0,143.0,143.0,143.0,143.0,...,143.0,143.0,143.0,143.0,143.0,143.0,143.0,143.0,143.0,143.0
Hydro_Large,408.1,408.1,408.1,408.1,408.1,408.1,408.1,408.1,408.1,408.1,...,408.1,408.1,408.1,408.1,461.0,461.0,461.0,461.0,461.0,461.0


### Total Annual Production by Technology

In [33]:
import pandas as pd
import plotly.graph_objects as go

# === Load technology color mapping ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load and rename production data ===
df_prod = pd.read_csv(res_dir / "ProductionByTechnologyAnnual.csv")
df_prod.rename(columns={
    "t": "TECHNOLOGY",
    "y": "YEAR",
    "ProductionByTechnologyAnnual": "VALUE"
}, inplace=True)

# === Pivot for area plot ===
df_pivot = df_prod.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
years = df_pivot.index.tolist()

# === Create figure ===
fig = go.Figure()
trace_names = []

# Add one stacked area trace per technology
for tech in df_pivot.columns:
    trace_names.append(tech)
    fig.add_trace(go.Scatter(
        x=years,
        y=df_pivot[tech].fillna(0),
        mode='lines',
        stackgroup='one',
        name=tech,
        line=dict(color=tech_colors.get(tech, "#D3D3D3")),  # Use consistent color or fallback
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Production: %{{y:.2f}} PJ"
    ))

# === Dropdown visibility controls ===
visible_all = [True] * len(trace_names)
visible_none = [False] * len(trace_names)
visible_no_prod = [not tech.endswith("_Prod") for tech in trace_names]

# === Layout and interactivity ===
fig.update_layout(
    title="Annual Electricity Production by Technology",
    xaxis_title="Year",
    yaxis_title="Production (PJ)",
    template="plotly_white",
    legend_title="Technology",
    height=600,
    hovermode="x unified",
    updatemenus=[
        {
            "buttons": [
                {"label": "All", "method": "update", "args": [{"visible": visible_all}]},
                {"label": "None", "method": "update", "args": [{"visible": visible_none}]},
                {"label": "Exclude _Prod", "method": "update", "args": [{"visible": visible_no_prod}]}
            ],
            "direction": "down",
            "showactive": True,
            "x": 1.05,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }
    ]
)

# Show plot
try:
    fig.show()
except ValueError:
    print("🔁 Fallback to browser due to rendering issue.")
    fig.show(renderer="browser")


In [ ]:
import pandas as pd

# === Load and rename production data ===
df_prod = pd.read_csv(res_dir / "ProductionByTechnologyAnnual.csv")
df_prod.rename(columns={
    "t": "TECHNOLOGY",
    "y": "YEAR",
    "ProductionByTechnologyAnnual": "VALUE"
}, inplace=True)

# === Convert PJ to MWh (1 PJ = 277,777.78 MWh)
PJ_TO_GWh = 1e15 / 3.6e9  # = 277.778
df_prod["VALUE"] = df_prod["VALUE"] * PJ_TO_GWh

# === Pivot to get technologies × years ===
df_production_mwh = df_prod.pivot_table(
    index="TECHNOLOGY",
    columns="YEAR",
    values="VALUE",
    aggfunc="sum"
).round(0)

# Set column and row names for clarity
df_production_mwh.index.name = "Technology (GWh)"
df_production_mwh.columns.name = "Year"

# === Display the converted table in MWh ===
display(df_production_mwh)


Year,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
Technology (MWh),,,,,,,,,,,,,,,,,,,,,
Biomass_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Biomass_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HFO_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12933.0,0.0,0.0,0.0,0.0,0.0,15985.0,26251.0,0.0,0.0
HFO_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,36949.0,0.0,0.0,0.0,0.0,0.0,45669.0,74999.0,0.0,0.0
Hydro_Bekongor,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8627.0,7666.0,0.0,348.0,5723.0,7788.0,5606.0,15768.0,28770.0,31594.0
Hydro_Betmai,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,434.0,...,99.0,0.0,0.0,0.0,394.0,325.0,394.0,798.0,1705.0,2493.0
Hydro_BumbunaI,2117.0,840.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2081.0,...,1351.0,0.0,0.0,0.0,3139.0,2300.0,1533.0,3285.0,5238.0,10877.0
Hydro_BumbunaII,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4698.0,...,2819.0,0.0,0.0,0.0,4176.0,3549.0,2505.0,6994.0,9030.0,9262.0
Hydro_Large,49945.0,24688.0,224167.0,38089.0,360431.0,24076.0,47725.0,370556.0,60588.0,78732.0,...,101680.0,98709.0,63671.0,1006534.0,854742.0,118431.0,95982.0,141886.0,253354.0,236186.0


### Fuel Consumption by Technology

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# === Load tech color map ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load fuel use data ===
df_use = pd.read_csv(res_dir / "UseByTechnology.csv")
df_use.rename(columns={
    "t": "TECHNOLOGY",
    "y": "YEAR",
    "UseByTechnology": "VALUE"
}, inplace=True)

# === Pivot data: fuel consumption by year/technology ===
df_fuel = df_use.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")

# === Create Plotly stacked area chart ===
fig = go.Figure()
for tech in df_fuel.columns:
    fig.add_trace(go.Scatter(
        x=df_fuel.index,
        y=df_fuel[tech].fillna(0),
        mode='lines',
        stackgroup='one',
        name=tech,
        line=dict(color=tech_colors.get(tech, "#D3D3D3")),  # fallback gray
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Fuel Used: %{{y:.2f}} PJ"
    ))

# === Layout tweaks ===
fig.update_layout(
    title="Annual Fuel Consumption by Technology",
    xaxis_title="Year",
    yaxis_title="Fuel Use (PJ)",
    template="plotly_white",
    legend_title="Technology",
    height=600,
    hovermode="x unified"
)

fig.show()

In [34]:
import pandas as pd

# === Load fuel use data ===
df_use = pd.read_csv(res_dir / "UseByTechnology.csv")
df_use.rename(columns={
    "t": "TECHNOLOGY",
    "y": "YEAR",
    "UseByTechnology": "VALUE"
}, inplace=True)

# === Convert from PJ to GWh (1 PJ = 277.778 GWh)
PJ_TO_GWh = 1e15 / 3.6e9  # 277.778
df_use["VALUE"] = df_use["VALUE"] * PJ_TO_GWh

# === Pivot table: rows = technologies, columns = years
df_fuel_use = df_use.pivot_table(
    index="TECHNOLOGY",
    columns="YEAR",
    values="VALUE",
    aggfunc="sum"
).round(1)

# Set row/column labels for clarity
df_fuel_use.index.name = "Technology"
df_fuel_use.columns.name = "Year"

# === Display the final table
display(df_fuel_use)


Year,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
Technology,,,,,,,,,,,,,,,,,,,,,
Biomass_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Biomass_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HFO_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,36948.7,0.0,0.0,0.0,0.0,0.0,45669.0,74999.4,0.0,0.0
HFO_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Bekongor,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Betmai,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_BumbunaI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_BumbunaII,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Large,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Annual Capital Costs by Technology

In [19]:
import pandas as pd
import plotly.graph_objects as go

# === Load technology color map ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load and prepare capital investment data ===
df_capex = pd.read_csv(res_dir / "CapitalInvestment.csv")
df_capex.rename(columns={"y": "YEAR", "t": "TECHNOLOGY", "CapitalInvestment": "VALUE"}, inplace=True)

# Pivot to wide format
df_pivot = df_capex.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
years = df_pivot.index.tolist()

# === Build figure ===
fig = go.Figure()

for tech in df_pivot.columns:
    fig.add_trace(go.Bar(
        x=years,
        y=df_pivot[tech].fillna(0),
        name=tech,
        marker=dict(color=tech_colors.get(tech, "#D3D3D3")),  # Fallback to gray
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Cost: %{{y:.2f}} Million USD"
    ))

# === Layout and interactivity ===
fig.update_layout(
    title="Capital Investment Cost by Technology",
    xaxis_title="Year",
    yaxis_title="Cost (Million USD)",
    barmode='stack',
    template="plotly_white",
    height=600,
    legend_title="Technology",
    hovermode="x unified",
    updatemenus=[
        {
            "buttons": [
                {"label": "All", "method": "update", "args": [{"visible": [True] * len(df_pivot.columns)}]},
                {"label": "None", "method": "update", "args": [{"visible": [False] * len(df_pivot.columns)}]},
            ],
            "direction": "down",
            "x": 1.05,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }
    ]
)

# === Show plot with fallback ===
try:
    fig.show()
except ValueError:
    print("🔁 Fallback to browser due to rendering issue.")
    fig.show(renderer="browser")

In [35]:
import pandas as pd

# === Load and prepare capital investment data ===
df_capex = pd.read_csv(res_dir / "CapitalInvestment.csv")
df_capex.rename(columns={"y": "YEAR", "t": "TECHNOLOGY", "CapitalInvestment": "VALUE"}, inplace=True)

# === Pivot table: rows = technologies, columns = years
df_capex_table = df_capex.pivot_table(
    index="TECHNOLOGY",
    columns="YEAR",
    values="VALUE",
    aggfunc="sum"
).round(1)

# Set index/column names for clarity
df_capex_table.index.name = "Technology"
df_capex_table.columns.name = "Year"

# === Display the table
display(df_capex_table)


Year,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
Technology,,,,,,,,,,,,,,,,,,,,,
Biomass_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Biomass_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HFO_Plant,66.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,27.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HFO_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Bekongor,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Betmai,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_BumbunaI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_BumbunaII,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Large,1224.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,158.7,0.0,0.0,0.0,0.0,0.0


### Annual Fixed Costs by Technology

In [20]:
import pandas as pd
import plotly.graph_objects as go

# === Load color mapping ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load and prepare data ===
df_fixed = pd.read_csv(res_dir / "AnnualFixedOperatingCost.csv")
df_fixed.rename(columns={"y": "YEAR", "t": "TECHNOLOGY", "AnnualFixedOperatingCost": "VALUE"}, inplace=True)

# Pivot for bar plotting
df_pivot = df_fixed.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
years = df_pivot.index.tolist()

# === Build stacked bar chart ===
fig = go.Figure()

for tech in df_pivot.columns:
    fig.add_trace(go.Bar(
        x=years,
        y=df_pivot[tech].fillna(0),
        name=tech,
        marker=dict(color=tech_colors.get(tech, "#D3D3D3")),  # fallback to light gray
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Fixed Cost: %{{y:.2f}} Million USD"
    ))

# === Layout ===
fig.update_layout(
    title="Annual Fixed O&M Cost by Technology",
    xaxis_title="Year",
    yaxis_title="Cost (Million USD)",
    barmode='stack',
    template="plotly_white",
    height=600,
    legend_title="Technology",
    hovermode="x unified",
    updatemenus=[
        {
            "buttons": [
                {"label": "All", "method": "update", "args": [{"visible": [True] * len(df_pivot.columns)}]},
                {"label": "None", "method": "update", "args": [{"visible": [False] * len(df_pivot.columns)}]},
            ],
            "direction": "down",
            "x": 1.05,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }
    ]
)

# Show the figure
try:
    fig.show()
except ValueError:
    print("🔁 Fallback to browser due to rendering issue.")
    fig.show(renderer="browser")

In [36]:
import pandas as pd

# === Load and prepare fixed O&M cost data ===
df_fixed = pd.read_csv(res_dir / "AnnualFixedOperatingCost.csv")
df_fixed.rename(columns={
    "y": "YEAR",
    "t": "TECHNOLOGY",
    "AnnualFixedOperatingCost": "VALUE"
}, inplace=True)

# === Pivot table: rows = technologies, columns = years
df_fixed_om_cost = df_fixed.pivot_table(
    index="TECHNOLOGY",
    columns="YEAR",
    values="VALUE",
    aggfunc="sum"
).round(1)

# Set row/column names for clarity
df_fixed_om_cost.index.name = "Technology"
df_fixed_om_cost.columns.name = "Year"

# === Display the table
display(df_fixed_om_cost)


Year,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
Technology,,,,,,,,,,,,,,,,,,,,,
Biomass_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Biomass_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HFO_Plant,3.1,3.1,3.1,3.1,3.1,3.9,3.9,3.9,3.9,3.9,...,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,3.6,1.6
HFO_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Bekongor,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.4,14.4,14.4,14.4,14.4,14.4,14.4,14.4,14.4,14.4
Hydro_Betmai,0.0,0.0,0.0,0.0,0.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
Hydro_BumbunaI,3.8,3.8,3.8,3.8,3.8,3.8,3.8,3.8,3.8,3.8,...,3.8,3.8,3.8,3.8,3.8,3.8,3.8,3.8,3.8,3.8
Hydro_BumbunaII,0.0,0.0,0.0,0.0,0.0,12.9,12.9,12.9,12.9,12.9,...,12.9,12.9,12.9,12.9,12.9,12.9,12.9,12.9,12.9,12.9
Hydro_Large,36.7,36.7,36.7,36.7,36.7,36.7,36.7,36.7,36.7,36.7,...,36.7,36.7,36.7,36.7,41.5,41.5,41.5,41.5,41.5,41.5


### Annual Variable Costs by Technology

In [21]:
import pandas as pd
import plotly.graph_objects as go

# === Load tech color scheme ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load and prepare variable cost data ===
df_var = pd.read_csv(res_dir / "AnnualVariableOperatingCost.csv")
df_var.rename(columns={
    "y": "YEAR",
    "t": "TECHNOLOGY",
    "AnnualVariableOperatingCost": "VALUE"
}, inplace=True)

# Pivot data for bar plot
df_pivot = df_var.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")
years = df_pivot.index.tolist()

# === Create interactive figure ===
fig = go.Figure()

for tech in df_pivot.columns:
    fig.add_trace(go.Bar(
        x=years,
        y=df_pivot[tech].fillna(0),
        name=tech,
        marker=dict(color=tech_colors.get(tech, "#D3D3D3")),  # fallback to light gray
        hovertemplate=f"{tech}<br>Year: %{{x}}<br>Variable Cost: %{{y:.2f}} Million USD"
    ))

# === Layout settings ===
fig.update_layout(
    title="Annual Variable Operating Cost by Technology",
    xaxis_title="Year",
    yaxis_title="Cost (Million USD)",
    barmode='stack',
    template="plotly_white",
    height=600,
    legend_title="Technology",
    hovermode="x unified",
    updatemenus=[
        {
            "buttons": [
                {"label": "All", "method": "update", "args": [{"visible": [True] * len(df_pivot.columns)}]},
                {"label": "None", "method": "update", "args": [{"visible": [False] * len(df_pivot.columns)}]},
            ],
            "direction": "down",
            "x": 1.05,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top"
        }
    ]
)

# === Show figure with fallback ===
try:
    fig.show()
except ValueError:
    print("🔁 Fallback to browser due to rendering issue.")
    fig.show(renderer="browser")

In [37]:
import pandas as pd

# === Load and prepare variable O&M cost data ===
df_var = pd.read_csv(res_dir / "AnnualVariableOperatingCost.csv")
df_var.rename(columns={
    "y": "YEAR",
    "t": "TECHNOLOGY",
    "AnnualVariableOperatingCost": "VALUE"
}, inplace=True)

# === Pivot table: rows = technologies, columns = years
df_variable_om_cost = df_var.pivot_table(
    index="TECHNOLOGY",
    columns="YEAR",
    values="VALUE",
    aggfunc="sum"
).round(1)

# Set labels
df_variable_om_cost.index.name = "Technology"
df_variable_om_cost.columns.name = "Year"

# === Display the table
display(df_variable_om_cost)


Year,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
Technology,,,,,,,,,,,,,,,,,,,,,
Biomass_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Biomass_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HFO_Plant,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HFO_Prod,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.2,0.0,0.0,0.0,0.0,0.0,1.5,2.5,0.0,0.0
Hydro_Bekongor,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Betmai,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_BumbunaI,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_BumbunaII,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hydro_Large,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Monthly Resolution Analysis

In [22]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

# === Load tech color map ===
with open("tech_colors.json") as f:
    tech_colors = json.load(f)

# === Load production data ===
df = pd.read_csv(res_dir / "ProductionByTechnology.csv")

# Rename columns for clarity
df.rename(columns={
    "t": "TECHNOLOGY",
    "l": "TIMESLICE",
    "f": "FUEL",
    "y": "YEAR",
    "ProductionByTechnology": "VALUE"
}, inplace=True)

# Filter electricity only
df = df[df["FUEL"] == "Electricity"]

# === Extract month from TIMESLICE (e.g. M01D -> 01) ===
df["MONTH"] = df["TIMESLICE"].str.slice(1, 3)

# Aggregate by month (summing DAY + NIGHT)
monthly_df = df.groupby(["YEAR", "MONTH", "TECHNOLOGY"])["VALUE"].sum().reset_index()

# Get sorted list of months
months = sorted(monthly_df["MONTH"].unique())
n_months = len(months)

# Setup subplots
fig = make_subplots(
    rows=n_months, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"Month {m}" for m in months],
    vertical_spacing=0.05
)

# Plot each month's production by technology
for i, month in enumerate(months):
    df_month = monthly_df[monthly_df["MONTH"] == month]
    pivot = df_month.pivot_table(index="YEAR", columns="TECHNOLOGY", values="VALUE", aggfunc="sum")

    for tech in pivot.columns:
        fig.add_trace(
            go.Bar(
                x=pivot.index,
                y=pivot[tech].fillna(0),
                name=tech,
                marker=dict(color=tech_colors.get(tech, "#D3D3D3")),
                hovertemplate=f"{tech}<br>Year: %{{x}}<br>Production: %{{y:.2f}} PJ",
                showlegend=(i == 0)
            ),
            row=i + 1, col=1
        )

# Layout
fig.update_layout(
    title="Monthly Electricity Production by Technology",
    height=300 * n_months,
    barmode="stack",
    template="plotly_white",
    legend_title="Technology",
    hovermode="x unified",
    showlegend=True
)

fig.update_xaxes(title_text="Year", tickangle=-30)
fig.update_yaxes(title_text="Production (PJ)")

fig.show()
